# SBA 7(a) Loan Risk — Probability Calibration

This notebook calibrates the predicted five-year charge-off
probabilities of the selected Logistic Regression and
Gradient Boosting models.

## Models

- Logistic Regression: 13 predictors, C = 1.0
- Gradient Boosting: 13 predictors, learning rate = 0.05,
  200 iterations, 31 maximum leaf nodes

## Chronological design

- FY2010–2015: Model training
- FY2016: Hyperparameter tuning
- FY2017: Probability calibration
- FY2018: Independent validation
- FY2019: Final testing

## Objectives

1. Assess calibration before adjustment.
2. Fit probability calibration using FY2017 only.
3. Preserve the frozen underlying models.
4. Evaluate calibrated predictions independently on FY2018.
5. Keep FY2019 untouched until final model selection.

## Methodological limitations

The modeling population is retrospective and status-selected.
The 13-feature predictor set remains provisional pending
complete approval-time feature provenance verification.

## Reproducibility note (repository review)

Run with Python 3.12.8 and scikit-learn 1.5.1, with the notebook working directory set to `notebooks/`. The large SQL-generated export is kept locally, rather than tracked by Git. Saved outputs below document historical runs; this reviewed copy was not re-executed.


In [1]:
import pandas as pd
import numpy as np
import joblib

from pathlib import Path

from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    roc_auc_score
)

pd.set_option("display.max_columns", None)

RANDOM_STATE = 42

DATA_PATH = Path("../data/exports/sba_model_export.csv")
MODEL_DIR = Path("../models")

# Fail with actionable guidance if this notebook is started from the wrong folder.
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Required SQL modeling export not found: {DATA_PATH}. "
        "Run this notebook from notebooks/ after generating data/exports/sba_model_export.csv."
    )

_required_models = (
    "logistic_regression_13_features.joblib",
    "gradient_boosting_13_features.joblib",
)
for _model_file in _required_models:
    if not (MODEL_DIR / _model_file).is_file():
        raise FileNotFoundError(
            f"Required saved model missing: {MODEL_DIR / _model_file}. "
            "Run the preceding modeling/calibration notebooks first."
        )


In [2]:
# Load the frozen 13-feature models
lr_model = joblib.load(
    MODEL_DIR / "logistic_regression_13_features.joblib"
)

gb_model = joblib.load(
    MODEL_DIR / "gradient_boosting_13_features.joblib"
)

print("Logistic Regression model loaded successfully.")
print("Gradient Boosting model loaded successfully.")

# Verify that the correct configurations were loaded
assert lr_model.named_steps["classifier"].C == 1.0

assert gb_model.learning_rate == 0.05
assert gb_model.max_iter == 200
assert gb_model.max_leaf_nodes == 31
assert gb_model.n_features_in_ == 13

print("\nSaved model configuration validation passed!")

Logistic Regression model loaded successfully.
Gradient Boosting model loaded successfully.

Saved model configuration validation passed!


In [3]:
# Load the original modeling export
df = pd.read_csv(
    DATA_PATH,
    parse_dates=["approval_date"],
    dtype={
        "sba_lender_id": "string",
        "lender_key": "string"
    }
)

# Select the reserved calibration cohort
calibrate_df = df.loc[
    df["modeling_period"] == "Calibrate"
].copy()

print("Full dataset shape:", df.shape)
print("Calibration dataset shape:", calibrate_df.shape)

print(
    "FY2017 charge-offs:",
    calibrate_df["chargeoff_within_5yr"].sum()
)

print(
    "FY2017 charge-off incidence:",
    f"{calibrate_df['chargeoff_within_5yr'].mean():.4%}"
)

# Validate cohort size and outcomes
assert df.shape == (478953, 25)
assert len(calibrate_df) == 56078

assert calibrate_df["chargeoff_within_5yr"].sum() == 2419
assert calibrate_df["approval_fy"].eq(2017).all()

print("\nFY2017 calibration cohort validation passed!")

Full dataset shape: (478953, 25)
Calibration dataset shape: (56078, 25)
FY2017 charge-offs: 2419
FY2017 charge-off incidence: 4.3136%

FY2017 calibration cohort validation passed!


In [4]:
numeric_features = [
    "gross_approval",
    "guarantee_pct",
    "initial_interest_rate_model",
    "jobs_supported"
]

categorical_features = [
    "business_age_group",
    "naics_sector",
    "project_state",
    "processing_method",
    "collateral_indicator",
    "revolver_status",
    "interest_rate_type",
    "business_type"
]

flag_features = [
    "invalid_interest_rate_flag"
]

feature_columns = (
    numeric_features
    + categorical_features
    + flag_features
)

target_column = "chargeoff_within_5yr"

print("Total predictors:", len(feature_columns))

assert len(feature_columns) == 13

Total predictors: 13


In [5]:
# Retrieve historical training records for category definitions
train_df = df.loc[
    df["modeling_period"] == "Train"
].copy()

# Prepare FY2017 predictors for Gradient Boosting
X_calibrate_gb = calibrate_df[feature_columns].copy()

# Store category definitions derived from the training period
category_mappings = {}

for feature in categorical_features:

    train_categories = pd.Index(
        train_df[feature].dropna().unique()
    )

    category_mappings[feature] = train_categories

    categorical_dtype = pd.CategoricalDtype(
        categories=train_categories
    )

    X_calibrate_gb[feature] = X_calibrate_gb[feature].astype(
        categorical_dtype
    )

print("FY2017 categorical dtypes:")

print(X_calibrate_gb[categorical_features].dtypes)

FY2017 categorical dtypes:
business_age_group      category
naics_sector            category
project_state           category
processing_method       category
collateral_indicator    category
revolver_status         category
interest_rate_type      category
business_type           category
dtype: object


In [6]:
# Verify all categorical columns have the correct dtype
assert all(
    isinstance(X_calibrate_gb[col].dtype, pd.CategoricalDtype)
    for col in categorical_features
)

# Verify feature order matches the fitted classifier
assert list(X_calibrate_gb.columns) == list(
    gb_model.feature_names_in_
)

# Count categorical values unseen during training
print("FY2017 unseen category counts:")

for feature in categorical_features:

    unseen_count = (
        calibrate_df[feature].notna()
        & X_calibrate_gb[feature].isna()
    ).sum()

    print(f"{feature}: {unseen_count}")

print("\nCalibration feature preparation validated!")

FY2017 unseen category counts:
business_age_group: 0
naics_sector: 0
project_state: 0
processing_method: 1
collateral_indicator: 0
revolver_status: 0
interest_rate_type: 0
business_type: 0

Calibration feature preparation validated!


In [7]:
# Prepare FY2017 predictors for Logistic Regression
X_calibrate_lr = calibrate_df[feature_columns].copy()

# Extract the actual FY2017 outcomes
y_calibrate = calibrate_df[target_column].copy()

# Verify dimensions
print("Logistic Regression features:", X_calibrate_lr.shape)
print("Gradient Boosting features:", X_calibrate_gb.shape)
print("Calibration outcomes:", y_calibrate.shape)

assert X_calibrate_lr.shape == (56078, 13)
assert X_calibrate_gb.shape == (56078, 13)
assert len(y_calibrate) == 56078

print("\nCalibration dataset shapes validated!")

Logistic Regression features: (56078, 13)
Gradient Boosting features: (56078, 13)
Calibration outcomes: (56078,)

Calibration dataset shapes validated!


In [8]:
# Generate FY2017 probabilities using the frozen models
lr_calibrate_probabilities = lr_model.predict_proba(
    X_calibrate_lr
)[:, 1]

gb_calibrate_probabilities = gb_model.predict_proba(
    X_calibrate_gb
)[:, 1]

# Verify probability outputs
for model_name, probabilities in [
    ("Logistic Regression", lr_calibrate_probabilities),
    ("Gradient Boosting", gb_calibrate_probabilities)
]:
    assert len(probabilities) == len(y_calibrate)
    assert np.isfinite(probabilities).all()
    assert ((probabilities >= 0) & (probabilities <= 1)).all()

    print(f"\n{model_name}")
    print(f"Minimum predicted probability: {probabilities.min():.4%}")
    print(f"Maximum predicted probability: {probabilities.max():.4%}")
    print(f"Average predicted probability: {probabilities.mean():.4%}")

print("\nBoth models' probability validation passed!")


Logistic Regression
Minimum predicted probability: 0.0081%
Maximum predicted probability: 20.5215%
Average predicted probability: 3.7409%

Gradient Boosting
Minimum predicted probability: 0.0633%
Maximum predicted probability: 43.0575%
Average predicted probability: 3.7002%

Both models' probability validation passed!


In [9]:
# Calculate training-period charge-off prevalence
train_prevalence = train_df[target_column].mean()

# Constant-probability reference based on historical training data
reference_probabilities = np.full(
    len(y_calibrate),
    train_prevalence
)

# Calculate Brier scores
reference_brier = brier_score_loss(
    y_calibrate,
    reference_probabilities
)

lr_brier = brier_score_loss(
    y_calibrate,
    lr_calibrate_probabilities
)

gb_brier = brier_score_loss(
    y_calibrate,
    gb_calibrate_probabilities
)

# Calculate Brier Skill Scores
lr_brier_skill = 1 - (lr_brier / reference_brier)
gb_brier_skill = 1 - (gb_brier / reference_brier)

# Display results
print(f"Training prevalence: {train_prevalence:.4%}")
print(f"FY2017 observed incidence: {y_calibrate.mean():.4%}")

print(f"\nReference Brier score: {reference_brier:.6f}")

print("\nLogistic Regression:")
print(f"Brier score: {lr_brier:.6f}")
print(f"Brier skill: {lr_brier_skill:.4f}")

print("\nGradient Boosting:")
print(f"Brier score: {gb_brier:.6f}")
print(f"Brier skill: {gb_brier_skill:.4f}")

# Validate numerical outputs
assert np.isfinite([
    reference_brier,
    lr_brier,
    gb_brier,
    lr_brier_skill,
    gb_brier_skill
]).all()

assert reference_brier > 0

print("\nUncalibrated Brier evaluation passed!")

Training prevalence: 3.7164%
FY2017 observed incidence: 4.3136%

Reference Brier score: 0.041311

Logistic Regression:
Brier score: 0.040456
Brier skill: 0.0207

Gradient Boosting:
Brier score: 0.040201
Brier skill: 0.0269

Uncalibrated Brier evaluation passed!


In [10]:
def calculate_calibration_deciles(y_true, probabilities):

    # Combine actual outcomes and predicted probabilities
    calibration_df = pd.DataFrame({
        "actual": np.asarray(y_true),
        "predicted": np.asarray(probabilities)
    })

    # Assign approximately equal-sized risk deciles
    calibration_df["decile"] = (
        pd.qcut(
            calibration_df["predicted"].rank(method="first"),
            q=10,
            labels=False
        ) + 1
    )

    # Calculate statistics for each decile
    summary = (
        calibration_df
        .groupby("decile")
        .agg(
            loan_count=("actual", "size"),
            chargeoffs=("actual", "sum"),
            avg_predicted_risk=("predicted", "mean"),
            observed_chargeoff_rate=("actual", "mean")
        )
        .reset_index()
    )

    # Difference between observed and predicted probabilities
    summary["calibration_gap"] = (
        summary["observed_chargeoff_rate"]
        - summary["avg_predicted_risk"]
    )

    return summary

In [11]:
# Calculate FY2017 calibration deciles
lr_calibration_deciles = calculate_calibration_deciles(
    y_calibrate,
    lr_calibrate_probabilities
)

gb_calibration_deciles = calculate_calibration_deciles(
    y_calibrate,
    gb_calibrate_probabilities
)

# Convert probability metrics to percentages for display
percentage_columns = [
    "avg_predicted_risk",
    "observed_chargeoff_rate",
    "calibration_gap"
]

print("Logistic Regression — FY2017 Calibration Deciles")

lr_display = lr_calibration_deciles.copy()
lr_display[percentage_columns] *= 100

display(lr_display.round(3))

print("\nGradient Boosting — FY2017 Calibration Deciles")

gb_display = gb_calibration_deciles.copy()
gb_display[percentage_columns] *= 100

display(gb_display.round(3))

Logistic Regression — FY2017 Calibration Deciles


,decile,loan_count,chargeoffs,avg_predicted_risk,observed_chargeoff_rate,calibration_gap
0,1,5608,31,0.543,0.553,0.010
1,2,5608,46,1.316,0.820,-0.496
2,3,5608,86,1.855,1.534,-0.321
3,4,5607,127,2.335,2.265,-0.070
4,5,5608,188,2.838,3.352,0.515
5,6,5608,234,3.374,4.173,0.799
6,7,5607,293,4.009,5.226,1.216
7,8,5608,400,4.861,7.133,2.272
8,9,5608,477,6.252,8.506,2.253
9,10,5608,537,10.025,9.576,-0.450



Gradient Boosting — FY2017 Calibration Deciles


,decile,loan_count,chargeoffs,avg_predicted_risk,observed_chargeoff_rate,calibration_gap
0,1,5608,27,0.553,0.481,-0.072
1,2,5608,52,1.099,0.927,-0.172
2,3,5608,71,1.595,1.266,-0.329
3,4,5607,143,2.107,2.550,0.443
4,5,5608,173,2.670,3.085,0.415
5,6,5608,238,3.293,4.244,0.951
6,7,5607,285,4.024,5.083,1.059
7,8,5608,354,4.978,6.312,1.335
8,9,5608,431,6.411,7.685,1.274
9,10,5608,645,10.272,11.501,1.230


In [12]:
for name, summary in [
    ("Logistic Regression", lr_calibration_deciles),
    ("Gradient Boosting", gb_calibration_deciles)
]:
    assert len(summary) == 10
    assert summary["loan_count"].sum() == 56078
    assert summary["chargeoffs"].sum() == 2419
    assert summary["decile"].tolist() == list(range(1, 11))

    print(f"{name}: Calibration deciles validated!")

print("\nAll decile validation checks passed!")

Logistic Regression: Calibration deciles validated!
Gradient Boosting: Calibration deciles validated!

All decile validation checks passed!


In [13]:
import sklearn
from sklearn.calibration import CalibratedClassifierCV

print("scikit-learn version:", sklearn.__version__)
print("Calibration class imported successfully!")

scikit-learn version: 1.5.1
Calibration class imported successfully!


In [14]:
# Calibrate the already-fitted Logistic Regression model
lr_calibrator = CalibratedClassifierCV(
    estimator=lr_model,
    method="sigmoid",
    cv="prefit"
)

lr_calibrator.fit(
    X_calibrate_lr,
    y_calibrate
)

print("Logistic Regression sigmoid calibration completed.")

# Calibrate the already-fitted Gradient Boosting model
gb_calibrator = CalibratedClassifierCV(
    estimator=gb_model,
    method="sigmoid",
    cv="prefit"
)

gb_calibrator.fit(
    X_calibrate_gb,
    y_calibrate
)

print("Gradient Boosting sigmoid calibration completed.")

# Verify each calibrator contains one fitted classifier/calibrator pair
assert len(lr_calibrator.calibrated_classifiers_) == 1
assert len(gb_calibrator.calibrated_classifiers_) == 1

print("\nBoth sigmoid calibrators fitted successfully!")

Logistic Regression sigmoid calibration completed.
Gradient Boosting sigmoid calibration completed.

Both sigmoid calibrators fitted successfully!


In [15]:
# Generate calibrated FY2017 probabilities
lr_calibrated_probabilities = lr_calibrator.predict_proba(
    X_calibrate_lr
)[:, 1]

gb_calibrated_probabilities = gb_calibrator.predict_proba(
    X_calibrate_gb
)[:, 1]

# Validate calibrated outputs
for model_name, probabilities in [
    ("Logistic Regression", lr_calibrated_probabilities),
    ("Gradient Boosting", gb_calibrated_probabilities)
]:
    assert len(probabilities) == len(y_calibrate)
    assert np.isfinite(probabilities).all()
    assert ((probabilities >= 0) & (probabilities <= 1)).all()

    print(f"\n{model_name}")
    print(f"Minimum calibrated probability: {probabilities.min():.4%}")
    print(f"Maximum calibrated probability: {probabilities.max():.4%}")
    print(f"Average calibrated probability: {probabilities.mean():.4%}")

print("\nBoth calibrated probability validations passed!")


Logistic Regression
Minimum calibrated probability: 0.0081%
Maximum calibrated probability: 23.8248%
Average calibrated probability: 4.3136%

Gradient Boosting
Minimum calibrated probability: 0.0614%
Maximum calibrated probability: 50.1201%
Average calibrated probability: 4.3136%

Both calibrated probability validations passed!


In [16]:
# Calculate Brier scores after sigmoid calibration
lr_calibrated_brier = brier_score_loss(
    y_calibrate,
    lr_calibrated_probabilities
)

gb_calibrated_brier = brier_score_loss(
    y_calibrate,
    gb_calibrated_probabilities
)

# Calculate relative Brier score improvements
lr_brier_improvement = (
    (lr_brier - lr_calibrated_brier) / lr_brier
) * 100

gb_brier_improvement = (
    (gb_brier - gb_calibrated_brier) / gb_brier
) * 100

# Calculate Brier Skill Scores against training-prevalence reference
lr_calibrated_skill = 1 - (
    lr_calibrated_brier / reference_brier
)

gb_calibrated_skill = 1 - (
    gb_calibrated_brier / reference_brier
)

# Build a comparison table
calibration_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Gradient Boosting"
    ],
    "Original_Brier": [
        lr_brier,
        gb_brier
    ],
    "Calibrated_Brier": [
        lr_calibrated_brier,
        gb_calibrated_brier
    ],
    "Brier_Improvement_Pct": [
        lr_brier_improvement,
        gb_brier_improvement
    ],
    "Calibrated_Brier_Skill": [
        lr_calibrated_skill,
        gb_calibrated_skill
    ]
})

display(calibration_comparison.round(6))

# Validate calculated metrics
assert np.isfinite(
    calibration_comparison.select_dtypes(
        include="number"
    ).to_numpy()
).all()

print("\nCalibration Brier comparison validated!")

,Model,Original_Brier,Calibrated_Brier,Brier_Improvement_Pct,Calibrated_Brier_Skill
0,Logistic Regression,0.040456,0.040429,0.067893,0.021367
1,Gradient Boosting,0.040201,0.040153,0.119246,0.028030



Calibration Brier comparison validated!


In [17]:
# Define a reusable ranking evaluation function
def evaluate_ranking(y_true, probabilities):

    y = np.asarray(y_true)
    probabilities = np.asarray(probabilities)

    review_count = int(np.ceil(len(y) * 0.10))

    top_indices = np.argsort(
        -probabilities,
        kind="stable"
    )[:review_count]

    captured = int(y[top_indices].sum())

    return {
        "Average_Precision": average_precision_score(
            y, probabilities
        ),
        "ROC_AUC": roc_auc_score(
            y, probabilities
        ),
        "Recall_at_10": captured / y.sum(),
        "Captured_Chargeoffs": captured
    }


# Evaluate original and calibrated model probabilities
ranking_results = []

for model_name, original, calibrated in [
    (
        "Logistic Regression",
        lr_calibrate_probabilities,
        lr_calibrated_probabilities
    ),
    (
        "Gradient Boosting",
        gb_calibrate_probabilities,
        gb_calibrated_probabilities
    )
]:

    for version, probabilities in [
        ("Original", original),
        ("Calibrated", calibrated)
    ]:

        metrics = evaluate_ranking(
            y_calibrate,
            probabilities
        )

        ranking_results.append({
            "Model": model_name,
            "Version": version,
            **metrics
        })


ranking_comparison = pd.DataFrame(ranking_results)

display(ranking_comparison.round(4))

,Model,Version,Average_Precision,ROC_AUC,Recall_at_10,Captured_Chargeoffs
0,Logistic Regression,Original,0.0861,0.7119,0.2220,537
1,Logistic Regression,Calibrated,0.0861,0.7119,0.2220,537
2,Gradient Boosting,Original,0.1002,0.7218,0.2666,645
3,Gradient Boosting,Calibrated,0.1002,0.7218,0.2666,645


In [18]:
# Define output paths
lr_calibrated_path = (
    MODEL_DIR / "logistic_regression_13_features_calibrated.joblib"
)

gb_calibrated_path = (
    MODEL_DIR / "gradient_boosting_13_features_calibrated.joblib"
)

# Save the calibrated models
joblib.dump(lr_calibrator, lr_calibrated_path)
joblib.dump(gb_calibrator, gb_calibrated_path)

# Verify saved files
print("Logistic Regression calibrator saved:", lr_calibrated_path.exists())
print("Gradient Boosting calibrator saved:", gb_calibrated_path.exists())

assert lr_calibrated_path.exists()
assert gb_calibrated_path.exists()

print("\nBoth calibrated models saved successfully!")

Logistic Regression calibrator saved: True
Gradient Boosting calibrator saved: True

Both calibrated models saved successfully!
